# bind_ddg — the binding-magnitude sensor, trained on the FULL SKEMPI set (~345 complexes)

The sandbox trained on only the 42 complexes it had structures for. This notebook does the real run on all of SKEMPI
(~345 complexes, ~7,000 measured ΔΔG-binding mutations) on your faster CPU + GPU, in two parts:

1. **Physics `bind_ddg`** (the production model) — structure-only physics + best-rotamer + charge features → the
   extrinsic magnitude sensor. Needs only PDB structures (fast). **This is the deliverable.**
2. **End-to-end surface test** at scale — physics vs the partner-aware geodesic surface encoder (fine-tuned end-to-end)
   vs both. The sandbox found this a dead lever (surf ~0.10, both < physics) because the WT surface is *mutation-blind*.
   This confirms it on all 345 with a GPU (needs dMaSIF clouds — the slower APBS build).

## 1 · GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device=='cuda' else 'CPU')

## 2 · Install the tools (APBS + PDB2PQR for clouds; sklearn/scipy for the models)

In [ ]:
import sys, subprocess, shutil
subprocess.run('apt-get -qq install -y apbs > /dev/null 2>&1', shell=True)
subprocess.run(sys.executable+' -m pip -q install pdb2pqr scikit-image biopython scikit-learn scipy 2>/dev/null', shell=True)
print('apbs:', shutil.which('apbs'), '| pdb2pqr:', shutil.which('pdb2pqr'))

## 3 · Get the code

In [ ]:
import os, sys
BRANCH='claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    os.system(f'git clone -q -b {BRANCH} https://github.com/nikku03/cell.git /content/cell')
else:
    os.system('cd /content/cell && git pull -q')
sys.path.insert(0,'/content/cell/colab')
print('code:', 'ok' if os.path.isdir('/content/cell/colab') else 'MISSING')

## 4 · Fetch all SKEMPI structures + train the PHYSICS model (the deliverable)
Downloads SKEMPI, fetches every complex's PDB, and trains `bind_ddg` held-out **by complex**, broken out by substitution
class (this is the honest all-AA number — no alanine flattering).

In [ ]:
import nexus_train as nt, flex_physics as fp, bind_ddg, importlib
for m in (nt, bind_ddg): importlib.reload(m)
CACHE='/content/skempi_cache'
nt.setup(CACHE)                                   # patches paths + downloads SKEMPI
skempi = sorted({r['pdb'] for r in fp.load_pdb_muts()})
print(len(skempi),'SKEMPI complexes; fetching PDBs ...')
got = nt.fetch(skempi, CACHE)
print(len(got),'PDBs present; training bind_ddg on the full set ...')
res = bind_ddg.train(got, CACHE)
print('\nPRODUCTION bind_ddg (held-out by complex):')
print('  all-AA r', res['r_all'], '| non-ala', res['r_nonala'], '| alanine', res['r_alanine'], '| hotspot AUC', res['hotspot_auc'])
print('  trained on', res['n_mutations'],'mutations /', res['n_complexes'],'complexes  (model -> outputs/orphan/bind_ddg_model.pkl)')

## 5 · Build dMaSIF clouds for all SKEMPI (for the end-to-end test)
APBS + marching-cubes surface per complex, parallel across CPU cores, cached/resumable. ~5–8 min on 12 cores.
(Skip this + the next cell if you only want the physics model.)

In [ ]:
t0=__import__('time').time()
data = nt.build(got, CACHE, workers=None)         # all cores; resumable
print(f'built/loaded {len(data)} surface clouds ({__import__("time").time()-t0:.0f}s)')

## 6 · End-to-end surface test at scale (GPU) — does the surface add binding-magnitude signal?
Trains three readouts held-out by complex: **phys** (baseline), **surf** (surface encoder alone, end-to-end),
**both**. If `both` beats `phys`, the surface adds signal; if not, the mutation-blind WT surface is confirmed a dead
lever for magnitude (belongs on Part-1 recognition, not Part-2 magnitude).

In [ ]:
import bind_ddg_e2e; importlib.reload(bind_ddg_e2e)
e = bind_ddg_e2e.run(CACHE, clouds_dir=CACHE+'/clouds', epochs=60, device=device)
print('\nSURFACE HELPS BINDING MAGNITUDE:', e['surface_helps_binding'])
print(e['verdict'])

## What you get
- **`bind_ddg`** trained on the full ~345-complex SKEMPI set — the production extrinsic magnitude sensor, with the honest
  held-out-by-complex number broken out by substitution class (`outputs/orphan/bind_ddg.json`, model `bind_ddg_model.pkl`).
- **A definitive answer** on the end-to-end surface lever at scale (`outputs/orphan/bind_ddg_e2e.json`) — whether a
  partner-aware structural encoder, fine-tuned end-to-end, adds anything over physics for *magnitude* (sandbox said no).

To keep the trained model, copy `bind_ddg_model.pkl` to Drive, or commit it from the checkout.